# Structure Learning for MNIST with Renormalised Generative Models

Implementation of the MNIST digit classification example from:
> Friston et al. (2025) *From pixels to planning: scale-free active inference.* Front. Netw. Physiol. 5:1521963

## Preprocessing Pipeline (Section: Image compression and compositionality)

From the paper (p.10):
> MNIST images were preprocessed by up-sampling to 32 pixels x 32 pixels, smoothing, and histogram equalization. In addition, they were converted into a format suitable for video processing with three (TrueColor or RGB) channels.

In [ ]:
import numpy as np
from jax import numpy as jnp
import matplotlib.pyplot as plt

from preprocess import load_mnist, preprocess
from utils import extract_exemplars

In [ ]:
x_train, y_train, x_test, y_test = load_mnist(cache_dir='./mnist_cache')

print(f"Training set: {x_train.shape}, labels: {y_train.shape}")
print(f"Test set:     {x_test.shape}, labels: {y_test.shape}")
print(f"Pixel range:  [{x_train.min():.1f}, {x_train.max():.1f}]")
print(f"Classes:      {np.unique(np.array(y_train))}")

In [ ]:
# Show a few raw 28x28 images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
fig.suptitle('Raw MNIST images (28x28)', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Preprocess

The full pipeline applies (in order):
1. **Gaussian smoothing** (sigma=0.3) - smooth before upsampling to reduce aliasing
2. **Histogram equalization** - enhance contrast
3. **Bilinear upsampling** 28x28 → 32x32 - makes image divisible into 4x4 patches for the blocking transformation
4. **Convert to RGB** (3 identical channels) - format suitable for video processing extension

Output shape: `(N, 3, 32, 32)`

see preprocess.py for how these functions are implemented.

## 3. Select Structure Learning Exemplars

From the paper (p.10):
> Based on the prior that there can be a dozen ways of writing any given number, the first 13 (Baker's dozen) images of each digit class were used for fast structure learning.

These 130 images (13 per class x 10 classes) are used to build the initial RGM hierarchy via fast structure learning. The remaining images are used for active learning.

In [ ]:
M_PER_CLASS = 13  # Baker's dozen
NUM_CLASSES = 10

x_exemplars_raw, y_exemplars, exemplar_idx = extract_exemplars(x_train, y_train, M_PER_CLASS, NUM_CLASSES)
print(f"Exemplar images: {x_exemplars_raw.shape}")
print(f"Per class: {[(y_exemplars == d).sum().item() for d in range(10)]}")

In [ ]:
# Preprocess the exemplars
x_exemplars = preprocess(x_exemplars_raw, target_size=32, sigma=0.3)
print(f"Preprocessed exemplars: {x_exemplars.shape}")

In [ ]:
# Display all 130 exemplars organised by class (cf. Figure 4 in the paper)
fig, axes = plt.subplots(NUM_CLASSES, M_PER_CLASS, figsize=(M_PER_CLASS, NUM_CLASSES))
for digit in range(NUM_CLASSES):
    mask = y_exemplars == digit
    digit_imgs = x_exemplars[mask]
    for j in range(M_PER_CLASS):
        axes[digit, j].imshow(digit_imgs[j, 0], cmap='gray')
        axes[digit, j].axis('off')
    axes[digit, 0].set_ylabel(str(digit), rotation=0, fontsize=12, labelpad=15)
fig.suptitle(f'Structure Learning Exemplars ({M_PER_CLASS} per class, preprocessed 32x32)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. SVD Discretisation

From the paper (p.8-10): Images are tessellated into 4x4 patches. Each patch location gets its own SVD basis learned from the structure exemplars. Continuous singular variates are quantised into 7 discrete levels, producing a discrete observation tensor per image. 

The main text in the paper says:

> "Each group is then subject to singular value decomposition, given a training set of images, to identify an orthogonal (spatial) basis set of singular vectors. This grouping is followed by a reduction operator that retains singular variates with large singular values (here, the first 32 principal vectors based on groups of 4 × 4 pixels)."

> "The set of singular variates for each group specifies the pattern for any given image at the corresponding location. The continuous variates can then be quantized to a discrete number of levels (here, seven)."

However the figure 4 caption says:

> "...in this example, the singular variates could take seven discrete values centered on zero for a maximum of 16 singular vectors."

And footnote 7 says:

> "In practice, we use overlapping groups, where the singular value decomposition is applied following weighting by a radial (Gaussian) basis function whose standard deviation is the distance between group centers."

In [ ]:
from discretise import DiscretiseConfig, compute_svd_basis, encode_images, decode_observations

config = DiscretiseConfig()
basis = compute_svd_basis(x_exemplars, config)
observations = encode_images(x_exemplars, basis, config)

print(f"SVD basis V:     {basis.V.shape}")
print(f"Bin edges:       {basis.bin_edges.shape}")
print(f"Observations:    {observations.shape}")
print(f"Value range:     [{observations.min()}, {observations.max()}]")

In [ ]:
# Reconstruct and compare (cf. Figure 4 right panel) - one of each digit
reconstructed = decode_observations(observations, basis, config)

# Pick the first exemplar of each digit (0-9)
show_idx = [jnp.where(y_exemplars == d)[0][0].item() for d in range(NUM_CLASSES)]

fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(NUM_CLASSES * 1.5, 3))
for col, idx in enumerate(show_idx):
    axes[0, col].imshow(x_exemplars[idx, 0], cmap='gray')
    axes[0, col].set_title(str(col), fontsize=11)
    axes[0, col].axis('off')
    axes[1, col].imshow(reconstructed[idx, 0], cmap='gray')
    axes[1, col].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('Reconstructed', fontsize=11)
fig.suptitle('SVD Discretisation: Original vs Reconstructed (7 levels, 16 components)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Build the RGM Hierarchy

From the paper (p.8-10): The hierarchy is built bottom-up from the structure exemplars:

1. **Level 1 (Patch states)**: Each patch location has its own set of **states** — the unique observation patterns seen across the structure exemplars. A batched pymdp Agent (one per patch) maps discrete SVD observations to these states via one-hot likelihoods.

2. **Levels 2+**: Each parent agent pools a 2×2 block of child MAP states. The unique child-state tuples observed across exemplars become the parent's state vocabulary. This halving continues until a single 1×1 grid remains — the image-level state.

The number of levels is derived automatically:
- `n_groups = image_size // group_size` → grid size at L1
- `n_levels = log2(n_groups) + 1` (here: 8×8 → 4×4 → 2×2 → 1×1 = 4 levels)

`RGMHierarchy.from_exemplars` performs all of these steps internally: SVD basis computation, patch state discovery, agent construction, and hierarchical inference at each level.

In [ ]:
from hierarchy import RGMHierarchy

rgm = RGMHierarchy.from_exemplars(x_exemplars, y_exemplars, config)

# Hierarchy summary
print(f"Number of levels: {len(rgm.levels)}")
for lv, level in enumerate(rgm.levels):
    grid = level.stats.num_states
    n_grid = grid.shape[0]
    print(f"  L{lv+1}: {n_grid}x{n_grid} grid, "
          f"states/location: {int(grid.min())}-{int(grid.max())} "
          f"(mean {float(grid.mean()):.1f})")

# Digit state map at the top level
n_top = int(rgm.levels[-1].stats.num_states[0, 0])
print(f"\nTop-level states: {n_top}")
print(f"Digit state map: {rgm.digit_state_map}")
n_unambiguous = (rgm.digit_state_map >= 0).sum()
print(f"Unambiguous states: {n_unambiguous}/{n_top}")
for d in range(10):
    states = [s for s in range(n_top) if rgm.digit_state_map[s] == d]
    print(f"  Digit {d}: {len(states)} states {states}")

In [ ]:
# Patch state count heatmap (Level 1)
l1_stats = rgm.levels[0].stats

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
im = ax.imshow(l1_stats.num_states, cmap='viridis')
ax.set_title('Unique states per patch location')
ax.set_xlabel('Patch column')
ax.set_ylabel('Patch row')
for i in range(l1_stats.num_states.shape[0]):
    for j in range(l1_stats.num_states.shape[1]):
        ax.text(j, i, int(l1_stats.num_states[i, j]), ha='center', va='center',
                color='white' if l1_stats.num_states[i, j] < l1_stats.num_states.max() * 0.7 else 'black',
                fontsize=8)
fig.colorbar(im, ax=ax, label='Number of unique states')
plt.tight_layout()
plt.show()

In [ ]:
from hierarchy import infer_patch_states, infer_hierarchical_states
from discretise import encode_images

# Encode exemplars and pick one image per digit
observations = encode_images(x_exemplars, rgm.basis, rgm.config)
show_idx = [jnp.where(y_exemplars == d)[0][0].item() for d in range(NUM_CLASSES)]

n_levels = len(rgm.levels)
fig, axes = plt.subplots(n_levels, NUM_CLASSES, figsize=(NUM_CLASSES * 1.5, n_levels * 1.5))
for col, idx in enumerate(show_idx):
    # L1: patch inference
    maps = infer_patch_states(rgm.levels[0].agent, rgm.levels[0].valid_mask, observations[idx])
    axes[0, col].imshow(maps, cmap='viridis')
    axes[0, col].set_title(str(col), fontsize=11)
    axes[0, col].axis('off')
    # L2+: hierarchical inference
    for lv, level in enumerate(rgm.levels[1:], 1):
        maps = infer_hierarchical_states(level.agent, level.valid_mask, maps)
        axes[lv, col].imshow(maps, cmap='viridis')
        axes[lv, col].axis('off')

for lv in range(n_levels):
    grid_size = rgm.levels[lv].stats.num_states.shape[0]
    axes[lv, 0].set_ylabel(f'L{lv+1} ({grid_size}x{grid_size})', fontsize=10)
fig.suptitle('Hierarchical MAP states across all levels', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Classification and Generation

The `RGMHierarchy` object supports two key operations:
- **classify**: bottom-up inference on unseen images → digit prediction
- **generate**: top-down state lookup from digit prior → reconstructed pixel image

In [ ]:
# Sanity check: classify the 130 structure exemplars (should be 100%)
pred_exemplars, l4_exemplars = rgm.classify(x_exemplars)
exemplar_acc = (pred_exemplars == np.array(y_exemplars)).mean()
print(f"Structure exemplar accuracy: {exemplar_acc:.1%} ({(pred_exemplars == np.array(y_exemplars)).sum()}/{len(y_exemplars)})")
if exemplar_acc < 1.0:
    mismatches = np.where(pred_exemplars != np.array(y_exemplars))[0]
    for idx in mismatches[:10]:
        print(f"  Image {idx}: true={y_exemplars[idx]}, pred={pred_exemplars[idx]}, L4 state={l4_exemplars[idx]}")

In [ ]:
# Classify first 100 test images
N_TEST = 100
x_test_pre = preprocess(x_test[:N_TEST], target_size=32, sigma=0.3)
y_test_sub = np.array(y_test[:N_TEST])

pred_test, l4_test = rgm.classify(x_test_pre)
test_acc = (pred_test == y_test_sub).mean()
print(f"Test accuracy ({N_TEST} images): {test_acc:.1%}")

# Per-digit accuracy
for d in range(10):
    mask = y_test_sub == d
    if mask.sum() > 0:
        d_acc = (pred_test[mask] == d).mean()
        print(f"  Digit {d}: {d_acc:.1%} ({(pred_test[mask] == d).sum()}/{mask.sum()})")

# Show grid with color-coded predictions
n_show = min(50, N_TEST)
cols = 10
rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.4))
for i, ax in enumerate(axes.flat):
    if i < n_show:
        ax.imshow(np.array(x_test_pre[i, 0]), cmap='gray')
        correct = pred_test[i] == y_test_sub[i]
        color = 'green' if correct else 'red'
        ax.set_title(f"{pred_test[i]}", fontsize=9, color=color)
    ax.axis('off')
fig.suptitle(f'Test Classification (green=correct, red=wrong) — {test_acc:.1%}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Generate one image per digit: argmax (top) and sampled (bottom)
rng = np.random.default_rng(42)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for d in range(10):
    img_argmax, _ = rgm.generate(digit=d, sample=False)
    axes[0, d].imshow(np.array(img_argmax[0, 0]), cmap='gray')
    axes[0, d].set_title(str(d), fontsize=11)
    axes[0, d].axis('off')

    img_sample, _ = rgm.generate(digit=d, sample=True, rng=rng)
    axes[1, d].imshow(np.array(img_sample[0, 0]), cmap='gray')
    axes[1, d].axis('off')

axes[0, 0].set_ylabel('Argmax', fontsize=10)
axes[1, 0].set_ylabel('Sampled', fontsize=10)
fig.suptitle('Top-down Generation: one image per digit', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Round-trip: generate per digit -> classify back
print("Round-trip: generate -> classify")
all_match = True
for d in range(10):
    img, _ = rgm.generate(digit=d, sample=False)
    pred, l4s = rgm.classify(img)
    match = pred[0] == d
    all_match = all_match and match
    status = "OK" if match else "MISMATCH"
    print(f"  Digit {d}: generated -> classified as {pred[0]} (L4 state {l4s[0]}) [{status}]")

print(f"Round-trip {'PASSED' if all_match else 'FAILED'}: all 10 digits")